In [4]:
# ===========================================================
# OpenAI Function Calling AI Agent
# 기능:
# 1. 만 나이 계산 (calculate_age)
# 2. 환율 변환 (convert_currency) -> amount만 사용 (환율 1,330원 고정)
# 3. BMI 계산 (calculate_bmi)
# ============================================================

# -----------------------------
# 1. 라이브러리 로드 및 초기화
# -----------------------------
from dotenv import load_dotenv  # .env 파일에서 환경 변수를 불러오기 위한 라이브러리
import os                       # 시스템 환경 변수에 접근하기 위한 표준 모듈
import json                     # JSON 형식의 데이터를 파이썬 딕셔너리로 변환하기 위한 모듈
from datetime import datetime   # 날짜 계산 및 포맷팅을 위한 모듈
from openai import OpenAI       # OpenAI API 호출을 위한 공식 클라이언트 라이브러리

# .env 파일에 저장된 환경 변수(API 키 등)를 불러옴
load_dotenv()

# 시스템 환경 변수에서 "OPENAI_API_KEY" 값을 가져와 변수에 저장
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# OpenAI 클라이언트 인스턴스를 생성하고 API 키를 전달
client = OpenAI(
    api_key=OPENAI_API_KEY
)

# -----------------------------
# 2. 요구사항 커스텀 함수 정의
# -----------------------------

# [기능 1] YYYY-MM-DD 형식의 생년월일을 받아 현재 기준 만 나이를 계산하는 함수
def calculate_age(birthdate: str) -> str:
    try:
        # 문자열 형태의 생년월일을 datetime 객체로 파싱합니다.
        birth_date = datetime.strptime(birthdate, "%Y-%m-%d")
        
        # 현재 날짜 및 시간을 가져옴
        today = datetime.today()
        
        # 만 나이 계산: (현재 연도 - 생년)에서, 올해 생일이 아직 지나지 않았으면 1을 뺀다
        age = today.year - birth_date.year - ((today.month, today.day) < (birth_date.month, birth_date.day))
        
        # 계산된 만 나이를 문자열로 반환
        return f"만 {age}세"
    except ValueError:
        # 생년월일 포맷이 올바르지 않을 때 예외 메시지를 반환
        return "날짜 형식이 올바르지 않습니다. (YYYY-MM-DD 형식 필요)"


# [기능 2] 달러 금액(amount)을 입력받아 고정 환율(1,330원)을 적용해 원화로 변환하는 함수
def convert_currency(amount: float) -> str:
    EXCHANGE_RATE = 1330  # 이미지 문제에 제시된 고정 환율 (1330원)
    
    # 전달받은 달러 금액에 고정 환율을 곱
    converted_amount = amount * EXCHANGE_RATE
    
    # 천 단위 쉼표(,)가 포함된 포맷으로 변환 결과 문자열을 반환
    return f"{amount}달러는 원화로 변환 시 {converted_amount:,.0f}원입니다. (적용 환율: {EXCHANGE_RATE}원)"


# [기능 3] 키(cm)와 몸무게(kg)를 입력받아 BMI를 계산하고 상태를 판정하는 함수
def calculate_bmi(height: float, weight: float) -> str:
    # 키 단위를 cm에서 m로 변환 (예: 170cm -> 1.7m)
    height_m = height / 100.0
    
    # BMI 공식: 몸무게(kg) / (키(m) * 키(m))
    bmi_value = weight / (height_m ** 2)
    
    # 구한 BMI 값에 따라 건강 상태 분류 기준을 적용
    if bmi_value < 18.5:
        status = "저체중"
    elif bmi_value < 23:
        status = "정상"
    elif bmi_value < 25:
        status = "과체중"
    else:
        status = "비만"
        
    # 소수점 둘째 자리까지 표현된 BMI 값과 판정 결과를 반환
    return f"BMI 지수는 {bmi_value:.2f}이며, '{status}' 상태입니다."


# -----------------------------
# 3. Function Calling Tools 정의 (OpenAI 모델 전달용)
# -----------------------------
tools = [
    {
        "type": "function", # 도구 타입 지정
        "name": "calculate_age", # 실행할 함수 이름
        "description": "생년월일(YYYY-MM-DD)을 입력받아 현재 기준의 만 나이를 계산한다.", # 함수 역할 설명
        "parameters": { # 매개변수 구조 정의
            "type": "object",
            "properties": {
                "birthdate": {
                    "type": "string", # 타입: 문자열
                    "description": "YYYY-MM-DD 형식의 생년월일 문자열" # 설명
                }
            },
            "required": ["birthdate"], # 필수 입력 인자
            "additionalProperties": False # 임의의 추가 인자 불허
        },
        "strict": True # Strict Mode 적용
    },
    {
        "type": "function",
        "name": "convert_currency",
        "description": "달러(USD) 금액을 입력받아 고정 환율 1330원을 적용해 원화(KRW)로 변환한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {
                    "type": "number", # 타입: 숫자 (수정사항 반영)
                    "description": "변환할 달러 금액 (숫자)"
                }
            },
            "required": ["amount"], # amount 인자만 필수
            "additionalProperties": False
        },
        "strict": True
    },
    {
        "type": "function",
        "name": "calculate_bmi",
        "description": "키(cm)와 몸무게(kg)를 입력받아 체질량지수(BMI)를 계산한다.",
        "parameters": {
            "type": "object",
            "properties": {
                "height": {
                    "type": "number", # 타입: 숫자 (cm)
                    "description": "키 (단위: cm)"
                },
                "weight": {
                    "type": "number", # 타입: 숫자 (kg)
                    "description": "몸무게 (단위: kg)"
                }
            },
            "required": ["height", "weight"], # 키, 몸무게 모두 필수
            "additionalProperties": False
        },
        "strict": True
    }
]


# -----------------------------
# 4. OpenAI Agent 클래스 정의
# -----------------------------
class OpenAIAgent:

    # 클래스 초기화 시 호출되는 생성자
    def __init__(self):
        print("=" * 60)
        print(" 과제수행: 3가지 커스텀 함수 호출 AI Agent ")
        print("=" * 60)

    # -----------------------------
    # GPT에게 질문을 보내 함수 호출 요구를 파악하는 메서드
    # -----------------------------
    def call_openai(self, user_input):
        # 모델에게 전달할 메시지 배열 구성
        input_messages = [
            {
                "role": "developer", # 개발자 프롬프트 (페르소나 지정)
                "content": "너는 나이 계산, 환율 변환, BMI 계산을 처리하는 에이전트이다. 반드시 제공된 툴을 사용하여 답변하라."
            },
            {
                "role": "user", # 터미널에서 받은 사용자 질문
                "content": user_input
            }
        ]

        # OpenAI Responses API 호출
        response = client.responses.create(
            model="gpt-4.1",       # 사용할 모델 지정
            input=input_messages,  # 메시지 전달
            tools=tools            # 함수 도구 정의 리스트 전달
        )
        return response # GPT 응답 객체 반환


    # -----------------------------
    # GPT가 요청한 함수를 실제 파이썬 환경에서 실행하는 메서드
    # -----------------------------
    def handle_function_call(self, response):
        outputs = [] # 함수 실행 결과들을 담을 배열 생성

        # response.output 내부의 모든 아이템 순회
        for item in response.output:
            # 항목 타입이 'function_call'이 아니면 건너뜀
            if item.type != "function_call":
                continue

            # GPT가 실행을 지정한 함수의 이름
            function_name = item.name
            
            # GPT가 추출한 JSON 문자열 형태의 인자값을 파이썬 딕셔너리로 변환
            arguments = json.loads(item.arguments)

            # 1. 나이 계산 함수 호출 분기
            if function_name == "calculate_age":
                result = calculate_age(
                    birthdate=arguments["birthdate"]
                )

            # 2. 환율 변환 함수 호출 분기 (amount만 전달)
            elif function_name == "convert_currency":
                result = convert_currency(
                    amount=arguments["amount"]
                )

            # 3. BMI 계산 함수 호출 분기
            elif function_name == "calculate_bmi":
                result = calculate_bmi(
                    height=arguments["height"],
                    weight=arguments["weight"]
                )
            
            # 예외 처리: 명시되지 않은 함수가 들어올 경우
            else:
                result = "지원하지 않는 함수입니다."

            # OpenAI가 규정한 출력 구조에 맞춰 결과 추가
            outputs.append(
                {
                    "type": "function_call_output",
                    "call_id": item.call_id, # 원래 매칭되었던 call_id 값 연결
                    "output": str(result)   # 함수 결과를 문자열 형태로 변환
                }
            )

        return outputs # 최종 결과를 담은 배열 반환


    # -----------------------------
    # 터미널 기반 상호작용 대화 루프 메서드
    # -----------------------------
    def chat(self):
        while True:
            # 사용자로부터 질문 문자열 입력 받기
            user_input = input("\n사용자 > ")

            # 'exit' 또는 'quit' 입력 시 대화 종료
            if user_input.lower() in ["exit", "quit"]:
                print("\n프로그램 종료")
                break

            # [Step 1] 사용자 입력으로 GPT 1차 호출 (어떤 함수를 실행할지 판단 받음)
            response = self.call_openai(user_input)

            # [Step 2] GPT의 호출 요구 사항을 바탕으로 파이썬 함수 실행
            outputs = self.handle_function_call(response)

            # 함수 호출이 불필요한 일반 대화 질문인 경우
            if len(outputs) == 0:
                print("\nGPT >", response.output_text)
                continue

            # [Step 3] 함수 실행 결과를 가지고 GPT 2차 호출 (결과를 바탕으로 문장 생성)
            final_response = client.responses.create(
                model="gpt-4.1",
                previous_response_id=response.id, # 이전 대화맥락 연결
                input=outputs                     # 함수 실행 결과 입력
            )

            # GPT가 다듬어낸 최종 답변 출력
            print("\nGPT >", final_response.output_text)


# -----------------------------
# 5. 메인 함수 정의 및 실행
# -----------------------------
def main():
    # 에이전트 인스턴스 생성
    agent = OpenAIAgent()
    
    # 루프 시작
    agent.chat()


# 파일이 직접 실행되었을 때만 main() 실행
if __name__ == "__main__":
    main()

 과제수행: 3가지 커스텀 함수 호출 AI Agent 

GPT > 2003년 11월 14일 생은 현재 만 22세입니다.

GPT > 2003년 11월 14일생은 만 22세입니다.

GPT > 100달러는 133,000원입니다. (적용 환율: 1,330원/달러 기준)

GPT > 키 170cm, 몸무게 65kg의 BMI는 22.49로, '정상' 상태입니다.

프로그램 종료
